# Decision Tree Example: Predicting Wide Receiver Draft Round

In this notebook, we will demonstrate the capabilities of the decision tree class in our ML package. To do so, we will take wide receiver's college football stats and predict the round of the NFL Draft they are chosen in.

In [35]:
# Import necessary libraries
import sys
import os

# Send Python to the project root so we can import our library
project_root = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/src"
sys.path.append(project_root)

# Import our ML library and other necessary libraries
import rice_ml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [36]:
# Import the dataset
data_path = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/data/nfl_draft_data.csv"
data = pd.read_csv(data_path)

Now that we have imported our data and package, we can prep the data for analysis.

In [37]:
# Filter the dataset to include only the relevant columns
data = data[["season", "pfr_player_name", "round", "position", "receptions","rec_yards","rec_tds"]]

# Only want Wide Receivers
data = data[data["position"] == "WR"]

# Only want players drafted this century
data = data[data["season"] >= 2000]
# Now we don't need the season column
data = data.drop(columns=["season"])

# Add column for yards per reception
data["yards_per_reception"] = data["rec_yards"] / data["receptions"]
# Add column for touchdowns per reception
data["tds_per_reception"] = data["rec_tds"] / data["receptions"]

# Drop rows with missing values
data = data.dropna()

As stated previously, we will only analyze wide receivers in this notebook. 

Furthermore, to better capture more recent trends, we eliminate all the players drafted before 2000. We deemed the year 2000 to be a fair balance between having a large enough sample size and capturing the recent stylistic developments in football. These developments include a much greater emphasis on the passing game, compared to older iterations of football.

Finally, we engineering two key features to better capture wide receiver efficiency. The NFL Draft is about getting players who will produce not players who have produced. Therefore, efficiency metrics, that likely better capture long run success, should be considered in our model.

Now we can standardize the data and train the decision tree model.

In [38]:
# Standardize the data using our StandardScaler class from our library
scaler = rice_ml.StandardScaler()
scaled_data = scaler.fit_transform(data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]])

# Combine scaled features with the target variable (round) into a new DataFrame
scaled_data = pd.DataFrame(scaled_data, columns=["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"])
scaled_data["round"] = data["round"].values

# Create a 80/20 train/test split with random shuffling
scaled_data = scaled_data.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(0.8 * len(scaled_data))
train_data = scaled_data.iloc[:train_size]
test_data = scaled_data.iloc[train_size:]

# Convert pandas DataFrames to numpy arrays for training and testing
X_train = train_data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]].values
y_train = train_data["round"].values
X_test = test_data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]].values
y_test = test_data["round"].values

# Train the decision tree model to predict the draft round based on the features
model = rice_ml.DecisionTree()
model.train(X_train, y_train)


Now that we have a trained model, we can generate predictions and evaluate the model's performance.

In [40]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Evaluate the model's performance using accuracy and mean absolute error from our library
accuracy = rice_ml.accuracy_score(y_test, predictions)
mae = rice_ml.mean_absolute_error(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")


Accuracy: 0.2042
Mean Absolute Error: 1.9789


Our model was able to predict 20% of the wide receiver's draft round correctly. This is a decent rate considering we built this model from just three simple statistical measures of wide receiver production. A mean absolute error of about 2, tells us that our predictions are off by an average of 2 rounds, soldifying the lack of strength in our model. Because the draft is just 7 rounds, an average error of 2 rounds is a bid knock on our model.

However, the model's limitations reveal the reality of the NFL Draft: film analysis is what makes or breaks a player's draft stock. Great production can certainly be beneficial to a player's draft stock, but it's overall impact is limited. 

This is likely because of the wide variety of college football team styles and levels. It's very possible for players with insignificant college production to perform exceptionally well in the NFL because they were limited by the context around their college team and boosted by their NFL team.

Ultimately, our decision tree didn't create a significantly strong model, but provided concrete evidence that the NFL Draft is ultimately an eye-test and skill evaluation game, rather than a production analysis game. 